<h1 align="center">
  <span style="background: linear-gradient(90deg, #1507d3ff, #7873f5); 
               -webkit-background-clip: text; 
               -webkit-text-fill-color: transparent;">
    AUTOMOBILE DATASET
  </span>
</h1>


---

Preparacion de datos 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter  
from torch.utils.data import random_split
from torch.utils.data import Dataset

In [ ]:
class AutomobileDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        # Define the column names based on the dataset
        cols = ["symboling", "normalized-losses", "make", "fuel-type", "aspiration", "num-of-doors", "body-style", 
                "drive-wheels", "engine-location", "wheel-base", "length", "width", "height", "curb-weight",
                "engine-type", "num-of-cylinders", "engine-size", "fuel-system", "bore", "stroke",
                "compression-ratio", "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "price"]       
        
        df_automobile = pd.read_csv(csv_file, na_values="?", names=cols)
        df_automobile.dropna(inplace=True)  
        
        # Convert categorical columns to numerical using one-hot encoding
        cols_categorical = ["make", "fuel-type", "aspiration", "num-of-doors", "body-style","drive-wheels", "engine-location", "engine-type", "num-of-cylinders", "fuel-system"]
        df_automobile = pd.get_dummies(df_automobile, columns=cols_categorical)

        # Separate features and target variable
        # axis = 1 means we are dropping a column, axis = 0 means we are dropping a row
        X = df_automobile.drop("price", axis=1).values
        y = df_automobile["price"].values

        # Standardize the features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Convert to PyTorch tensors
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        # view(-1, 1) means we want to reshape the tensor to have one column and as many rows as needed
        y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

        self.data = torch.cat((X_scaled, y_tensor), 1)

        self.root_dir = root_dir
        self.transform = transform

        def __len__(self):
            return len(self.data)
        
        def __getitem__(self, idx):
            if torch.is_tensor(idx):
                idx = idx.tolist()
            
            preds = self.data[idx, :-1]  # All columns except the last one are features
            spcs = self.data[idx, -1]     # The last column is the target variable (price)
            
            sample= (preds, spcs)

            if self.transform:
                sample = self.transform(sample)
            return sample
        

In [ ]:
dataset = AutomobileDataset(csv_file="../../docs/automobile/imports-85.data", root_dir="./")
print(f"Total samples in dataset: {len(dataset)}")
display(dataset[0])

<class 'pandas.DataFrame'>
Index: 159 entries, 3 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          159 non-null    int64  
 1   normalized-losses  159 non-null    float64
 2   make               159 non-null    str    
 3   fuel-type          159 non-null    str    
 4   aspiration         159 non-null    str    
 5   num-of-doors       159 non-null    str    
 6   body-style         159 non-null    str    
 7   drive-wheels       159 non-null    str    
 8   engine-location    159 non-null    str    
 9   wheel-base         159 non-null    float64
 10  length             159 non-null    float64
 11  width              159 non-null    float64
 12  height             159 non-null    float64
 13  curb-weight        159 non-null    int64  
 14  engine-type        159 non-null    str    
 15  num-of-cylinders   159 non-null    str    
 16  engine-size        159 non-null    int64  

In [ ]:
class AutomobileModel(nn.Module):
    def __init__(self, input_size):
        super(AutomobileModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        #TODO !! The first layer takes the input size (number of features) and outputs 64 neurons
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        #The forward method defines how the data flows through the network
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
model = AutomobileModel(input_size=dataset.data.shape[1] - 1)  # Number of features is total columns - 1 (for price)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
display(model)

In [ ]:
tamTrain = int(0.8 * len(dataset))
tamTest = len(dataset) - tamTrain
train_dataset, test_dataset = random_split(dataset, [tamTrain, tamTest])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
model = AutomobileModel(5)
optimizer = optim.Adam(model.parameters(), lr=0.001)

inputPrueba, dest = next(iter(train_loader))
print(f"Input shape: {inputPrueba.shape}, Target shape: {dest.shape}")

loss = loss_fn(model(inputPrueba), dest)
print(f"Initial loss: {loss.item()}")


In [ ]:
def train_one_epoch(model, train_loader, optimizer, loss_fn):
    model.train()
    last_loss = 0.0
    # usamos enumerate para saber en que batch imos
    for i, data in enumerate(train_loader):
        # Every data instance is an input + label pair
        inputs, labels = data
        # Zero your gradients for every batch!
        optimizer.zero_grad()
        # Make predictions for this batch
        outputs = model(inputs)
        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        # Adjust learning weights
        optimizer.step()
        # Gather data and report
        running_loss += loss.item()
        if i % 10 == 9:
            last_loss = running_loss / 10 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.
    return last_loss

In [ ]:
EPOCHS = 100
loss_list     = torch.zeros((EPOCHS,))
accuracy_list = torch.zeros((EPOCHS,))

# Crear o SummaryWriter para TensorBoard 
writer = SummaryWriter()

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch + 1))

    # Poñemos o modelo en modo entrenamento
    model.train(True)
    avg_loss = train_one_epoch(epoch, writer)
    loss_list[epoch] = avg_loss
    # Non se precisan os gradientes para o test
    model.train(False)

In [ ]:
plt.style.use('ggplot')
fig, (ax2) = plt.subplots(1, figsize=(12, 6), sharex=True)
ax2.plot(loss_list)
ax2.set_ylabel("validation loss")
ax2.set_xlabel("epochs")

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(range(1, EPOCHS + 1), loss_list.numpy(), color='#2e28cc', linewidth=2.5, label='Loss de entrenamiento')

if EPOCHS >= 5:
    loss_series = pd.Series(loss_list.numpy())
    smooth_loss = loss_series.rolling(window=5, min_periods=1).mean()
    ax.plot(range(1, EPOCHS + 1), smooth_loss, color='#ff7f0e', linewidth=2, linestyle='--', label='Media móvil (5)')

best_epoch = torch.argmin(loss_list).item() + 1
best_loss = torch.min(loss_list).item()

ax.scatter(best_epoch, best_loss, color='red', s=80, zorder=5, label=f'Mínimo: época {best_epoch}')
ax.annotate(
    f'Época {best_epoch}\nLoss = {best_loss:.4f}',
    xy=(best_epoch, best_loss),
    xytext=(best_epoch + 3, best_loss + 10),
    arrowprops=dict(arrowstyle='->', color='red'),
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='red', alpha=0.8)
)

ax.set_title('Evolución de la pérdida durante el entrenamiento', fontsize=16, fontweight='bold')
ax.set_xlabel('Épocas', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()